# 01 — Análise Exploratória e Entendimento do Problema

**Tech Challenge Fase 3 — FIAP Pós-Tech IA Scientist**

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # permite `import src` a partir de notebooks/
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src import config as C
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

## 1. Contexto e pergunta analítica

O Compromisso Nacional Criança Alfabetizada fixa a meta de 80% de crianças alfabetizadas ao fim do 2º ano até 2030.
O Indicador Criança Alfabetizada (ICA), construído a partir do SAEB 2º ano, mede anualmente a taxa por município.

**Pergunta:** dado o que sabemos sobre um município em *t* (resultado do ano anterior, território, condições socioeconômicas e estrutura da rede), qual a probabilidade de ele estar **em risco** (taxa < 60%) em *t+1* — e quais fatores explicam esse risco?

**Desenho temporal:** features de 2023 → target de 2024. Nenhuma variável de 2024 entra como feature (ver `src/data/build_dataset.py`).

## 2. Hipóteses a validar nesta EDA

| # | Hipótese | Como testar aqui |
|---|---|---|
| H1 | Inércia: a taxa de 2023 é o preditor mais forte; a maioria dos municípios em risco em 2024 já estava em risco em 2023 | matriz de transição 2023→2024; correlação taxa_t × taxa_t1 |
| H2 | Desigualdade territorial: Norte/Nordeste concentram o risco mesmo controlando por IDHM/renda | risco por região; boxplot de taxa por região dentro de faixas de IDHM |
| H3 | Condição socioeconômica: IDHM-E, renda e CadÚnico explicam parte da variância não explicada pela taxa anterior | correlação parcial (resíduo de taxa_t1 ~ taxa_t vs variáveis socioeconômicas) |
| H4 | Estrutura da rede: infraestrutura, docentes com superior e distorção idade-série associam-se ao risco | correlações e boxplots por status de risco |
| H5 | Participação baixa no SAEB em t → maior risco e maior variância em t+1 | risco por decil de participação; desvio da variação por decil |
| H6 | Porte: municípios pequenos têm maior volatilidade de taxa entre anos | |variação| por porte populacional |

In [ ]:
silver = pd.read_parquet(C.RAW_DIR / "municipio_silver.parquet")
silver = silver[silver[C.COL_REDE] == C.REDE]
print(silver.shape); silver.groupby(C.COL_ANO)[C.COL_ID].nunique()

## 3. Distribuições e qualidade dos dados

In [ ]:
silver.describe().T.round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for ano, cor in ((2023, "#7f8c8d"), (2024, "#2e86c1")):
    sns.histplot(silver.loc[silver[C.COL_ANO] == ano, C.COL_TAXA], bins=40, ax=ax[0], color=cor, label=str(ano), alpha=.6)
ax[0].axvline(C.CORTE_RISCO, color="#c0392b", ls="--", label=f"corte risco {C.CORTE_RISCO:.0f}%")
ax[0].legend(); ax[0].set_title("Distribuição da taxa de alfabetização por ano")
(silver.isna().mean() * 100).sort_values(ascending=False).head(12).plot.barh(ax=ax[1]); ax[1].set_title("% missing por coluna")
plt.tight_layout()

## 4. H1 — Inércia e matriz de transição 2023 → 2024

In [ ]:
dados = pd.read_parquet(C.PROCESSED_DIR / "dataset_modelagem.parquet")
from src.visualization.plots import plot_matriz_transicao
tab = plot_matriz_transicao(dados, f"{C.COL_TAXA}_t", "taxa_alfabetizacao_t1", C.CORTE_RISCO, C.IMAGES_DIR / "03_matriz_transicao.png")
print("correlação taxa_t × taxa_t1:", dados[[f"{C.COL_TAXA}_t", "taxa_alfabetizacao_t1"]].corr().iloc[0, 1].round(3))
tab

> **Leitura:** os municípios que *mudam* de estado (fora da diagonal) são exatamente os que o modelo precisa capturar além da inércia. Registre aqui a proporção e onde eles se concentram.

## 5. H2 — Território

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
dados.groupby("regiao")[C.TARGET].mean().sort_values().plot.barh(ax=ax[0], color="#c0392b"); ax[0].set_title("Fração de municípios em risco (2024) por região")
sns.boxplot(data=dados, x="regiao", y=f"{C.COL_TAXA}_t", ax=ax[1]); ax[1].set_title("Taxa 2023 por região")
plt.tight_layout()

In [ ]:
# H2 controlando por IDHM (requer fonte Atlas)
if "atlas_idhm" in dados:
    dados["faixa_idhm"] = pd.qcut(dados["atlas_idhm"], 4, labels=["Q1 baixo", "Q2", "Q3", "Q4 alto"])
    display(dados.pivot_table(index="regiao", columns="faixa_idhm", values=C.TARGET, aggfunc="mean").round(2))
else:
    print("Atlas ainda não carregado — rode src.data.download_external")

## 6. H3/H4 — Socioeconômico e estrutura da rede: correlação com o *resíduo* da inércia

In [ ]:
# Resíduo: o que a taxa de 2023 NÃO explica da taxa de 2024
from sklearn.linear_model import LinearRegression
m = LinearRegression().fit(dados[[f"{C.COL_TAXA}_t"]], dados["taxa_alfabetizacao_t1"])
dados["residuo_inercia"] = dados["taxa_alfabetizacao_t1"] - m.predict(dados[[f"{C.COL_TAXA}_t"]])
externas = [c for c in dados.columns if c.startswith(("atlas_", "pib_", "inep_", "censo_", "ideb_", "fundeb_", "cadunico_", "ibge_"))]
if externas:
    corr = dados[externas + ["residuo_inercia", C.TARGET]].corr()[["residuo_inercia", C.TARGET]].drop(["residuo_inercia", C.TARGET])
    display(corr.sort_values(C.TARGET).round(3))
else:
    print("sem variáveis externas ainda")

## 7. H5 — Participação e H6 — Porte

In [ ]:
dados["decil_particip"] = pd.qcut(dados[f"{C.COL_PARTICIP}_t"], 10, labels=False, duplicates="drop")
dados["variacao"] = dados["taxa_alfabetizacao_t1"] - dados[f"{C.COL_TAXA}_t"]
g = dados.groupby("decil_particip").agg(risco=(C.TARGET, "mean"), desvio_variacao=("variacao", "std"), n=(C.COL_ID, "size"))
display(g.round(3))
if "porte" in dados:
    display(dados.groupby("porte").agg(risco=(C.TARGET, "mean"), abs_variacao=("variacao", lambda s: s.abs().mean()), n=(C.COL_ID, "size")).round(3))

## 8. Correlações entre features (multicolinearidade — informa a escolha de modelos lineares vs árvores)

In [ ]:
num = dados.select_dtypes("number").drop(columns=[C.COL_ID, C.TARGET, "taxa_alfabetizacao_t1"], errors="ignore")
plt.figure(figsize=(14, 11)); sns.heatmap(num.corr(), cmap="coolwarm", center=0, vmin=-1, vmax=1); plt.title("Correlação entre features (bloco A + externas)")

## 9. Conclusões da EDA → decisões de modelagem

Preencher após executar com dados reais:

- **H1:** ...
- **H2:** ...
- **H3/H4:** ...
- **H5:** ...
- **H6:** ...

**Decisões:** (a) manter o bloco histórico como features e treinar também a variante *sem histórico* para isolar fatores estruturais; (b) usar PR-AUC e recall da classe risco como métricas principais; (c) tratar missing com imputação por mediana + indicador; (d) categorias (UF/região/porte) via one-hot no pipeline.